In [1]:
## verify the similarity matrix performance

In [4]:
import pickle as pkl
import pandas as pd 
import numpy as np
import os

In [5]:
path = '../extracted_data/'
os.listdir(path)

['AnimeList.csv',
 'UserAnimeList.csv',
 'UserList.csv',
 'anime_cleaned.csv',
 'anime_filtered.csv',
 'animelists_cleaned.csv',
 'animelists_filtered.csv',
 'users_cleaned.csv',
 'users_filtered.csv']

In [31]:
anime_sim_matrix = pkl.load(open('../heavy_models/collaborative_model_sim_matrix.pkl', 'rb'))

In [41]:
animes = pkl.load(open('../models/collaborative_filtering_movies.pkl','rb'))

In [88]:
animes.head(10)

,anime_id,title
0,1,Cowboy Bebop
1,6,Trigun
2,7,Witch Hunter Robin
3,8,Beet the Vandel Buster
4,15,Eyeshield 21
5,16,Hachimitsu to Clover
6,17,Hungry Heart: Wild Striker
7,18,Initial D Fourth Stage
8,19,Monster
9,20,Naruto


In [89]:
anime_sim_matrix

array([[1.        , 0.68255327, 0.39863052, ..., 0.01897271, 0.        ,
        0.00569343],
       [0.68255327, 1.        , 0.40790983, ..., 0.014522  , 0.        ,
        0.00998856],
       [0.39863052, 0.40790983, 1.        , ..., 0.01058849, 0.        ,
        0.        ],
       ...,
       [0.01897271, 0.014522  , 0.01058849, ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.00569343, 0.00998856, 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(3540, 3540))

In [94]:
def recommend(anime_name, top_n=5):
    # Get the row corresponding to the anime title
    row = animes[animes['title'] == anime_name]
    
    if row.empty:
        print("❌ Anime not found.")
        return

    # Get index and anime_id
    index = row.index[0]  # exact position in the DataFrame
    anime_id = row['anime_id'].values[0]

    print(f"📌 Found '{anime_name}' at index {index} with anime_id {anime_id}")

    # Zip anime IDs with similarity scores
    similarities = anime_sim_matrix[index]
    zipped = list(zip(animes['anime_id'], similarities))

    # Sort by similarity score (excluding the anime itself)
    zipped = sorted(zipped, key=lambda x: x[1], reverse=True)
    zipped = [z for z in zipped if z[0] != anime_id]  # remove the anime itself

    # Display top recommendations
    print(f"\n🎯 Top {top_n} similar anime:")
    for i in range(top_n):
        rec_id = zipped[i][0]
        score = zipped[i][1]
        rec_title = animes[animes['anime_id'] == rec_id]['title'].values[0]
        print(f"{i+1}. {rec_title} (Score: {score:.4f})")


In [97]:
recommend('Sword Art Online')

📌 Found 'Sword Art Online' at index 1892 with anime_id 11757

🎯 Top 5 similar anime:
1. Shingeki no Kyojin (Score: 0.8643)
2. Sword Art Online II (Score: 0.8527)
3. Angel Beats! (Score: 0.8461)
4. No Game No Life (Score: 0.8430)
5. Toradora! (Score: 0.8154)


In [111]:
anime_sim_matrix = pkl.load(open('../heavy_models/collaborative_model_sim_matrix.pkl', 'rb'))

top_k = 10
top_sim_dict = {}

for idx, row in enumerate(anime_sim_matrix):
    # Get top 10 indices (excluding self)
    top_indices = np.argsort(row)[::-1]  # descending sort
    top_indices = top_indices[top_indices != idx][:top_k]  # exclude self

    top_sim_dict[idx] = [(int(i), float(row[i])) for i in top_indices]

In [112]:
with open('../models/top_10_sim_collaborative_filtering.pkl', 'wb') as f:
    pkl.dump(top_sim_dict, f)